In [1]:
%load_ext autoreload
%autoreload 2

In [14]:
import os
import sys
import json
from pathlib import Path

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from utils.plots import calculate_accuracy, plot_confusion_matrices, plot_distributions, plot_accuracy, plot_metric_heatmap, plot_precision_recall_curves, plot_cooccurrence, plot_error_distribution, GT_COLS, PRED_COLS, LABELS
from utils.tools import get_config, collect_jobs, prettify_table

In [15]:
def create_row_annotation(config):
    annotations = []

    if bool(config.get("use_instructions", 0)):
        annotations.append("with instructions")
    else:
        annotations.append("without instructions")

    n_demos = config.get("number_of_demonstrations")
    annotations.append(f"number of demonstrations: {n_demos}")
    if n_demos > 0:
        t = ""
        demo_type = config.get("type_of_demonstrations")
        match demo_type:
            case -1:
                t = "negative"
            case 0:
                t = "mixed"
            case 1:
                t = "positive"

        annotations.append(f"type of demonstrations: {t}")


    return "\n".join(annotations)

In [16]:
finished_jobs = collect_jobs("./outputs").get("Qwen2.5-7B-Instruct")

#finished_jobs = finished_jobs[:2]

def conf_matrices(jobs):
    num_jobs = len(jobs)
    
    print("Number of completed jobs: " + str(num_jobs))
    
    fig, axes = plt.subplots(num_jobs, 3, figsize=(16, 5 * num_jobs))
    if num_jobs == 1:
        axes = [axes]
    
    fig.suptitle(jobs[0]["model"], fontsize=16)
    
    for i, job_info in enumerate(jobs):
        config = get_config(job_info["config_json"])
            
        df = pd.read_csv(job_info["result_csv"], sep=";")
        plot_confusion_matrices(df, axes[i], use_title=True if i == 0 else False, xlabel="Predicted" if i == (num_jobs - 1) else "", ylabel="True")
    
        axes[i][0].annotate(
            create_row_annotation(config),
            xy=(-0.2, 0.5),
            xycoords='axes fraction',
            fontsize=12,
            ha='right',
            va='center',
            rotation=90
        )
    return fig, axes

#fig, axes = conf_matrices(finished_jobs)
#fig.tight_layout(rect=[0, 0, 1, 0.98])
#plt.savefig("conf.png", pad_inches=0.3)
#plt.show()

In [18]:
# TODO: Inspect prompts of all runs!!!

def accuracy_table(jobs):
    cols = ["model", "use_instructions", "type_of_demonstrations", "number_of_demonstrations", "theme_accuracy", "topic_accuracy", "concept_accuracy", "total_accuracy"]
    table = {col_name: [] for col_name in cols}
    for job in jobs:
        df = pd.read_csv(job["result_csv"], sep=";")

        config = get_config(job["config_json"])

        table["model"].append(job["model"])
        table["use_instructions"].append(config["use_instructions"])
        table["type_of_demonstrations"].append(config["type_of_demonstrations"])
        table["number_of_demonstrations"].append(config["number_of_demonstrations"])
        
        for name, score in calculate_accuracy(df).items():
            table[name].append(score)

    return table

res = pd.DataFrame(accuracy_table(finished_jobs))
res = prettify_table(res)
res

,model,use_instructions,type_of_demonstrations,number_of_demonstrations,theme_accuracy,topic_accuracy,concept_accuracy,total_accuracy
1,Qwen2.5-7B-Instruct,yes,mixed,0,0.844860,0.878505,0.846729,0.800000
0,Qwen2.5-7B-Instruct,no,negative,1,0.659813,0.680374,0.613084,0.416822
3,Qwen2.5-7B-Instruct,yes,negative,1,0.620561,0.661682,0.605607,0.274766
2,Qwen2.5-7B-Instruct,no,positive,1,0.652336,0.665421,0.633645,0.543925
4,Qwen2.5-7B-Instruct,yes,positive,1,0.747664,0.785047,0.600000,0.411215
5,Qwen2.5-7B-Instruct,no,negative,6,0.734579,0.736449,0.760748,0.650467
8,Qwen2.5-7B-Instruct,yes,negative,6,0.685981,0.719626,0.618692,0.353271
6,Qwen2.5-7B-Instruct,no,mixed,6,0.728972,0.775701,0.728972,0.659813
9,Qwen2.5-7B-Instruct,yes,mixed,6,0.811215,0.841121,0.607477,0.478505
7,Qwen2.5-7B-Instruct,no,positive,6,0.616822,0.689720,0.648598,0.532710


In [6]:
def build_figure_for_csv(csvs,
                         labels=LABELS,
                         cols1=GT_COLS,
                         cols2=PRED_COLS,
                         conf=True, acc=True, dist=True):

    filepath, model = csvs
    
    df = pd.read_csv(filepath, sep=";")

    # Layout:
    # Row 0: confusion matrices (3 plots)
    # Row 1: accuracy (1 plot spanning all columns)
    # Row 2-4: distributions (3 rows x 2 columns)

    num_rows = 0
    if conf:
        num_rows += 1
    if acc:
        num_rows += 1
    if dist:
        num_rows += 3

    fig = plt.figure(figsize=(16, num_rows * 6))
    gs = fig.add_gridspec(num_rows, 3)

    # --- Confusion matrices ---
    if conf:
        cm_axes = [
            fig.add_subplot(gs[0, i]) for i in range(3)
        ]
        plot_confusion_matrices(df, cm_axes, labels, cols1, cols2)

    # --- Accuracy (span full width) ---
    if acc:
        acc_ax = fig.add_subplot(gs[1 if conf else 0, :])
        plot_accuracy(df, acc_ax, cols1, cols2)

    # --- Distributions ---
    if dist:
        dist_axes = [
            [fig.add_subplot(gs[num_rows - 3 + i, 0]),
             fig.add_subplot(gs[num_rows - 3 + i, 1])]
            for i in range(len(cols1))
        ]
    
        plot_distributions(df,
                           pd.DataFrame(dist_axes).values,
                           labels,
                           cols1, cols2)

    fig.suptitle(model, fontsize=16)
    fig.tight_layout()

    return fig


def run_on_csvs(csv_paths, conf=True, acc=True, dist=True):
    figs = []

    for path in csv_paths:
        fig = build_figure_for_csv(path, conf=conf, acc=acc, dist=dist)
        figs.append(fig)

    return figs

#run_on_csvs(csvs, acc=False, dist=False)